# QueueMind: self-hosted decision model (Laya) on Colab

**Laya** (Convai Innovations, Apache-2.0) is an open-weight *decision model*, an open alternative to TypeSafe's Jev. Instead of generating text, it answers **typed questions** (choice / score / yes-no) about a piece of text with **calibrated probabilities**, in ~33 ms on a T4.

QueueMind uses it as **System 1** for walk-in triage: *which department? how urgent? possible emergency?*
- Confident answer (≥ 0.75) → used directly, no LLM call.
- Not confident → the LLM fast lane decides (**System 2**).
- Red-flag rules and the safety ratchet still apply: priority can only go up.

`laya-serve` speaks Jev's `/v1/systemone` API, so **Kev** (open, Qwen-based, Jev-compatible) can be dropped in instead.

**How it connects:** model server → free Cloudflare tunnel (HTTPS) → the notebook announces the URL to the app (`/api/runtime`) and sends a heartbeat every 30 s. Stop the notebook and the app falls back to the LLM on its own within ~90 s.

**Before running:** `Runtime → Change runtime type → T4 GPU`, and add Colab secrets (🔑 icon, enable *Notebook access*):
- `APP_URL`: e.g. `https://queuemind-demo.vercel.app`
- `RUNTIME_SECRET`: same value as the app's `RUNTIME_SECRET`

In [ ]:
RUN_LAYA = True     # decision model (System 1). This is the point of the notebook.
RUN_LLAMA = False   # optional: self-hosted Llama for SMS drafts + agent last-resort fallback
LLAMA_MODEL = "llama3.1:8b"
from google.colab import userdata
import subprocess, os, time, re, json, threading, requests
APP_URL = userdata.get("APP_URL").rstrip("/")
RUNTIME_SECRET = userdata.get("RUNTIME_SECRET")
print("App:", APP_URL, "| Laya:", RUN_LAYA, "| Llama:", RUN_LLAMA)

## 1 · Laya decision server

In [ ]:
if RUN_LAYA:
    !pip -q install "laya[serve]"
    # typed-decisions = the checkpoint fine-tuned for workflow questions (base ones are near chance zero-shot).
    laya_env = dict(os.environ, LAYA_DEVICE="cuda", LAYA_PRELOAD="1", LAYA_MODELS="typed-decisions", LAYA_DEFAULT_MODEL="typed-decisions")
    laya_proc = subprocess.Popen(["laya-serve"], env=laya_env, stdout=open("laya.log", "w"), stderr=subprocess.STDOUT)
    LAYA_LOCAL = "http://localhost:8000"
    for _ in range(200):   # first start downloads the checkpoint (~1.6 GB)
        try:
            if requests.get(f"{LAYA_LOCAL}/health", timeout=2).ok: break
        except Exception:
            time.sleep(3)
    print(open("laya.log").read()[-1500:])

In [ ]:
# Self-test with the exact questions QueueMind asks for every walk-in
def ask(state):
    t = time.time()
    r = requests.post(f"{LAYA_LOCAL}/v1/systemone", json={"state": state, "model": "typed-decisions", "questions": {
        "specialty": {"type": "choice", "instructions": "Which clinic department should see this patient?",
                      "criteria": {"General Medicine": "adults: fever, cough, pain, infections, chronic conditions, prescriptions, check-ups",
                                   "Pediatrics": "children, babies and infants: any symptom in a child"}},
        "urgency": {"type": "score", "instructions": "How urgently does this patient need to be seen?", "criteria": ["routine", "soon", "emergency"]},
        "emergency": {"type": "noul", "instructions": "Does this describe a possible medical emergency, such as chest pain, trouble breathing, stroke signs, heavy bleeding or loss of consciousness?"}}}, timeout=60)
    print(f"{(time.time()-t)*1000:.0f} ms |", state); print(json.dumps(r.json().get("answers"), indent=1))

if RUN_LAYA:
    ask("Crushing chest pain and sweating for 20 minutes")
    ask("My 4 year old has had an ear ache since last night")
    ask("Need a repeat prescription for my asthma inhaler")

## 2 · Optional: self-hosted Llama (Ollama)

In [ ]:
if RUN_LLAMA:
    !sudo apt-get -qq install -y zstd > /dev/null 2>&1
    !curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1
    env = dict(os.environ, OLLAMA_HOST="0.0.0.0:11434", OLLAMA_ORIGINS="*", OLLAMA_KEEP_ALIVE="24h",
               OLLAMA_CONTEXT_LENGTH="16384", OLLAMA_NUM_PARALLEL="1", OLLAMA_FLASH_ATTENTION="1")
    ollama = subprocess.Popen(["ollama", "serve"], env=env, stdout=open("ollama.log", "w"), stderr=subprocess.STDOUT)
    time.sleep(5)
    !ollama pull {LLAMA_MODEL}

## 3 · Tunnels + registration

In [ ]:
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared

def open_tunnel(port):
    p = subprocess.Popen(["./cloudflared", "tunnel", "--url", f"http://localhost:{port}", "--http-host-header", f"localhost:{port}", "--no-autoupdate"],
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    deadline = time.time() + 60
    while time.time() < deadline:
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", p.stdout.readline())
        if m: return m.group(0)
    raise RuntimeError(f"tunnel for port {port} did not start")

ENDPOINTS = []   # (name the app knows, public url, model label)
if RUN_LAYA:  ENDPOINTS.append(("laya", open_tunnel(8000), "laya"))
if RUN_LLAMA: ENDPOINTS.append(("colab", open_tunnel(11434), LLAMA_MODEL))
time.sleep(5)
for name, url, model in ENDPOINTS: print(f"{name:6s} {url}  ({model})")

def heartbeat():
    while True:
        for name, url, model in ENDPOINTS:
            try:
                r = requests.post(f"{APP_URL}/api/runtime", json={"secret": RUNTIME_SECRET, "name": name, "url": url, "model": model,
                                                                  "meta": {"gpu": "T4", "host": "colab"}}, timeout=15)
                print(time.strftime("%H:%M:%S"), name, r.status_code, end="  ")
            except Exception as e:
                print(name, "heartbeat failed:", e)
        print(end="")
        time.sleep(30)
threading.Thread(target=heartbeat, daemon=True).start()
print("Registered. Keep this tab open.")

In [ ]:
# 4 · What the app sees (re-run any time)
time.sleep(3)
s = requests.get(f"{APP_URL}/api/status", timeout=30).json()
print(json.dumps({"decision": s.get("decision"), "selfHostedLlama": s["selfHosted"], "fastLaneOrder": s["fastLaneOrder"]}, indent=2))